In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.loader import DataLoader
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path
from tqdm.notebook import tqdm
import time

from halide_gnn_cost_model.data import PipelineDataset
from halide_gnn_cost_model.model import PipeGCN, PipeGAT, PipelineModel

In [ ]:
PIPELINES_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/pipelines-16k-data")
GCN_MODELS_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gcn")
GAT_MODELS_DIR = Path("/Users/fangjun/Documents/stanford/cs224w/halide-gnn-cost-model/resources/models/gat")
# Create models directory if it doesn't exist
GCN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
GAT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
summary_writer = SummaryWriter(log_dir="resources/runs/" + time.strftime("%Y%m%d-%H%M%S"))

In [ ]:
USE_GPU = True

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
# Load dataset
dataset = PipelineDataset(PIPELINES_DIR)
data = dataset[0]  # Get the first pipeline graph
data.metadata()

In [ ]:
# Train/test split
num_train = int(0.95 * len(dataset))
train_dataset, test_dataset = random_split(dataset, [num_train, len(dataset) - num_train])
len(train_dataset), len(test_dataset)

# GCN

In [ ]:
DIM_EMBEDDING = 64
gcn = PipeGCN(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")

In [ ]:
model = PipelineModel(gcn, DIM_EMBEDDING, 5, len(dataset.ast_vocab), len(dataset.sched_vocab))
model = model.to(device)
model

In [ ]:
data_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
data_loader

In [ ]:
# Save model every n epochs
SAVE_EVERY = 5
NUM_EPOCHS = 5

# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.MSELoss()

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(data_loader):
        optimizer.zero_grad()
        batch = batch.to(device)
        pred = model(batch, batch["loop_level"].ptr)
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), torch.log(batch.y))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        summary_writer.add_scalar("Loss/train", loss.item(), epoch * len(data_loader) + batch_idx)
    # Print average loss for the epoch
    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    # Save model
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(model.state_dict(), GCN_MODELS_DIR / f"pipeline_model_epoch_{epoch+1}.pt")

## Eval

In [ ]:
def average_runtime_error(model, data_loader):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            pred = model(batch, batch["loop_level"].ptr)
            error_rates = torch.abs(pred.reshape(-1) - batch.y) / batch.y
            total_error += torch.mean(error_rates).item()
    return total_error / len(data_loader)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, test_loader) * 100:.2f}%")

In [ ]:
# Plot predicted vs actual runtimes
import matplotlib.pyplot as plt
all_preds = []
all_trues = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = torch.exp(model(batch, batch["loop_level"].ptr))
        all_preds.append(pred.reshape(-1).cpu())
        all_trues.append(batch.y.cpu())
all_preds = torch.cat(all_preds)
all_trues = torch.cat(all_trues)
plt.scatter(all_trues, all_preds, alpha=0.5)
plt.plot([all_trues.min(), all_trues.max()], [all_trues.min(), all_trues.max()], 'r--')
plt.xscale("log")
plt.yscale("log")
plt.xlabel("True Runtime")
plt.ylabel("Predicted Runtime")
plt.title("Predicted vs True Runtimes")
plt.show()

In [ ]:
idx = 0  # Index of the sample to inspect
model.eval()
with torch.no_grad():
    sample_data = test_dataset[idx]
    sample_data = sample_data.to(device)
    pred_runtime = torch.exp(model(sample_data))
    print(f"Predicted runtime: {pred_runtime}, True runtime: {sample_data.y}")

# GAT

In [ ]:
DIM_EMBEDDING = 64
gcn = PipeGAT(hidden_channels=DIM_EMBEDDING, out_channels=DIM_EMBEDDING, num_layers=4)
gcn = to_hetero(gcn, data.metadata(), aggr="sum")

In [ ]:
model = PipelineModel(gcn, DIM_EMBEDDING, 5, len(dataset.ast_vocab), len(dataset.sched_vocab))
model = model.to(device)
model

In [ ]:
data_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
data_loader

In [ ]:
# Save model every n epochs
SAVE_EVERY = 5
NUM_EPOCHS = 5

# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.MSELoss()

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(data_loader):
        optimizer.zero_grad()
        batch = batch.to(device)
        pred = model(batch, batch["loop_level"].ptr)
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), torch.log(batch.y))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        summary_writer.add_scalar("Loss/train", loss.item(), epoch * len(data_loader) + batch_idx)
    # Print average loss for the epoch
    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    # Save model
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(model.state_dict(), GAT_MODELS_DIR / f"pipeline_model_epoch_{epoch+1}.pt")

## Eval

In [ ]:
def average_runtime_error(model, data_loader):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            pred = model(batch, batch["loop_level"].ptr)
            error_rates = torch.abs(pred.reshape(-1) - batch.y) / batch.y
            total_error += torch.mean(error_rates).item()
    return total_error / len(data_loader)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"Average Runtime Error on Test Set: {average_runtime_error(model, test_loader) * 100:.2f}%")

In [ ]:
# Plot predicted vs actual runtimes
import matplotlib.pyplot as plt
all_preds = []
all_trues = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = torch.exp(model(batch, batch["loop_level"].ptr))
        all_preds.append(pred.reshape(-1).cpu())
        all_trues.append(batch.y.cpu())
all_preds = torch.cat(all_preds)
all_trues = torch.cat(all_trues)
plt.scatter(all_trues, all_preds, alpha=0.5)
plt.plot([all_trues.min(), all_trues.max()], [all_trues.min(), all_trues.max()], 'r--')
plt.xscale("log")
plt.yscale("log")
plt.xlabel("True Runtime")
plt.ylabel("Predicted Runtime")
plt.title("Predicted vs True Runtimes")
plt.show()

In [ ]:
idx = 0  # Index of the sample to inspect
model.eval()
with torch.no_grad():
    sample_data = test_dataset[idx]
    sample_data = sample_data.to(device)
    pred_runtime = torch.exp(model(sample_data))
    print(f"Predicted runtime: {pred_runtime}, True runtime: {sample_data.y}")

# Graph Transformer